# 09 — FINAL MobileNetV2 Grad-CAM Explainability

**NO TRAINING IN THIS NOTEBOOK.**

Purpose: address Reviewer #10.1 by generating Grad-CAM visualisations for the **final scientifically valid MobileNetV2 frozen-backbone baseline**.

The notebook automatically selects representative examples from the final clean test set:

1. Correct cataract — high-confidence example
2. Correct normal — high-confidence example
3. Cataract failure case
4. Normal failure case
5. Additional difficult clinical example

For every selected image it saves:

- original image
- Grad-CAM heatmap
- Grad-CAM overlay
- true class
- predicted class
- class probability
- source test filepath
- PNG at 300 dpi
- PDF container for publication workflow

The Grad-CAM target is the final MobileNetV2 spatial feature representation (`out_relu`, with fallback logic if required).

**Important:** Grad-CAM is an interpretability aid, not evidence that the model has learned a clinically causal feature.

In [ ]:
# ============================================================
# CELL 1 — SETUP
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

PROJECT = Path('/content/drive/MyDrive/Cataract')

MODEL_DIR = (
    PROJECT
    / 'FINAL_REVISION_2026_08'
    / 'reviewer_10_2_clean_split'
    / 'MobileNetV2_frozen'
)

OUT = (
    PROJECT
    / 'FINAL_REVISION_2026_08'
    / 'final_gradcam_mobilenetv2'
)

OUT.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / 'best.keras'
INDEX_PATH = MODEL_DIR / 'test_predictions_index.csv'
PROBS_PATH = MODEL_DIR / 'probs.npy'
Y_TRUE_PATH = MODEL_DIR / 'y_true.npy'
Y_PRED_PATH = MODEL_DIR / 'y_pred.npy'

CLASS_NAMES = [
    'Cataract',
    'Normal',
    'Not Eye'
]

for p in [
    MODEL_PATH,
    INDEX_PATH,
    PROBS_PATH,
    Y_TRUE_PATH,
    Y_PRED_PATH
]:
    assert p.exists(), f'STOP: Missing {p}'

print('TensorFlow:', tf.__version__)
print('Model:', MODEL_PATH)
print('Output:', OUT)
print('\n✅ CELL 1 COMPLETE')

In [ ]:
# ============================================================
# CELL 2 — LOAD FINAL MODEL + FINAL TEST ARRAYS
# ============================================================

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

index_df = pd.read_csv(
    INDEX_PATH
)

probs = np.load(
    PROBS_PATH
)

y_true = np.load(
    Y_TRUE_PATH
)

y_pred = np.load(
    Y_PRED_PATH
)

assert probs.shape == (2587, 3)
assert y_true.shape == (2587,)
assert y_pred.shape == (2587,)
assert len(index_df) == 2587

assert np.array_equal(
    y_pred,
    probs.argmax(axis=1)
)

assert np.array_equal(
    y_true,
    index_df['y_true'].to_numpy()
)

assert np.array_equal(
    y_pred,
    index_df['y_pred'].to_numpy()
)

print('Model layers:', len(model.layers))
print('Prediction shape:', probs.shape)
print('Test rows:', len(index_df))

print('\nFinal MobileNetV2 test accuracy:')
print(
    f'{100 * np.mean(y_true == y_pred):.4f}%'
)

print('\n✅ CELL 2 COMPLETE')

In [ ]:
# ============================================================
# CELL 3 — LOCATE THE FINAL BACKBONE SPATIAL FEATURE LAYER
# ============================================================

preferred_names = [
    'out_relu',
    'Conv_1_bn',
    'Conv_1'
]

target_layer = None

for name in preferred_names:
    try:
        candidate = model.get_layer(name)

        if len(candidate.output.shape) == 4:
            target_layer = candidate
            break
    except Exception:
        pass

if target_layer is None:

    # Fallback: find the last 4-D layer before the custom head.
    # Avoid head layers by stopping before the first obvious head Conv2D.
    four_d = []

    for layer in model.layers:
        try:
            shape = layer.output.shape

            if len(shape) == 4:
                four_d.append(layer)
        except Exception:
            pass

    if not four_d:
        raise RuntimeError(
            'STOP: No spatial feature layer found.'
        )

    target_layer = four_d[-1]

print('Grad-CAM target layer:')
print('Name:', target_layer.name)
print('Class:', target_layer.__class__.__name__)
print('Output shape:', target_layer.output.shape)

grad_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=[
        target_layer.output,
        model.output
    ]
)

print('\n✅ GRAD-CAM TARGET LAYER READY')
print('✅ CELL 3 COMPLETE')

In [ ]:
# ============================================================
# CELL 4 — GRAD-CAM FUNCTIONS
# ============================================================

def load_image_for_model(filepath):

    img = tf.keras.utils.load_img(
        filepath,
        target_size=(224, 224),
        color_mode='rgb',
        interpolation='nearest'
    )

    arr = tf.keras.utils.img_to_array(
        img
    )

    original = arr.astype(
        np.uint8
    )

    x = (
        arr.astype(np.float32)
        / 255.0
    )

    return (
        original,
        np.expand_dims(x, axis=0)
    )


def make_gradcam_heatmap(
    img_batch,
    class_index
):

    with tf.GradientTape() as tape:

        feature_maps, predictions = grad_model(
            img_batch,
            training=False
        )

        class_score = predictions[
            :,
            class_index
        ]

    grads = tape.gradient(
        class_score,
        feature_maps
    )

    if grads is None:
        raise RuntimeError(
            'STOP: Gradients are None.'
        )

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0, 1, 2)
    )

    feature_maps = feature_maps[0]

    heatmap = tf.reduce_sum(
        feature_maps
        * pooled_grads,
        axis=-1
    )

    heatmap = tf.nn.relu(
        heatmap
    )

    max_value = tf.reduce_max(
        heatmap
    )

    if float(max_value) > 0:
        heatmap = (
            heatmap
            / max_value
        )

    return heatmap.numpy()


def resize_heatmap(
    heatmap,
    height,
    width
):

    hm = tf.image.resize(
        heatmap[..., np.newaxis],
        (height, width),
        method='bilinear'
    )

    return np.squeeze(
        hm.numpy()
    )


def overlay_heatmap(
    original,
    heatmap,
    alpha=0.40
):

    h, w = original.shape[:2]

    hm = resize_heatmap(
        heatmap,
        h,
        w
    )

    cmap = plt.get_cmap(
        'jet'
    )

    colored = (
        cmap(hm)[..., :3]
        * 255
    ).astype(np.uint8)

    overlay = (
        (1 - alpha)
        * original.astype(np.float32)
        +
        alpha
        * colored.astype(np.float32)
    )

    return np.clip(
        overlay,
        0,
        255
    ).astype(np.uint8)


print('✅ Grad-CAM functions ready')
print('✅ CELL 4 COMPLETE')

In [ ]:
# ============================================================
# CELL 5 — SELECT REPRESENTATIVE TEST EXAMPLES
# ============================================================

df = index_df.copy()

df['confidence'] = probs[
    np.arange(len(probs)),
    y_pred
]

df['true_probability'] = probs[
    np.arange(len(probs)),
    y_true
]

df['correct'] = (
    y_true == y_pred
)

df['row_index'] = np.arange(
    len(df)
)

selected = []


def add_case(
    case_name,
    subset,
    sort_col,
    ascending=False
):

    if len(subset) == 0:
        print(
            f'⚠️ No candidate found for {case_name}'
        )
        return

    row = (
        subset
        .sort_values(
            sort_col,
            ascending=ascending
        )
        .iloc[0]
    )

    selected.append({
        'Case': case_name,
        'row_index': int(
            row['row_index']
        )
    })


# 1 — Correct cataract, highest confidence
add_case(
    'Correct_Cataract_HighConfidence',
    df[
        (df['y_true'] == 0)
        &
        (df['y_pred'] == 0)
    ],
    'confidence',
    ascending=False
)

# 2 — Correct normal, highest confidence
add_case(
    'Correct_Normal_HighConfidence',
    df[
        (df['y_true'] == 1)
        &
        (df['y_pred'] == 1)
    ],
    'confidence',
    ascending=False
)

# 3 — Cataract failure
cat_fail = df[
    (df['y_true'] == 0)
    &
    (df['y_pred'] != 0)
]

add_case(
    'Cataract_Failure',
    cat_fail,
    'confidence',
    ascending=False
)

# 4 — Normal failure
normal_fail = df[
    (df['y_true'] == 1)
    &
    (df['y_pred'] != 1)
]

add_case(
    'Normal_Failure',
    normal_fail,
    'confidence',
    ascending=False
)

# 5 — difficult but correctly classified clinical image:
# lowest correct-class confidence among Cat/Normal.
difficult_correct = df[
    (df['correct'])
    &
    (df['y_true'].isin([0, 1]))
]

add_case(
    'Difficult_Correct_Clinical',
    difficult_correct,
    'true_probability',
    ascending=True
)

selected_df = pd.DataFrame(
    selected
)

selected_df = selected_df.merge(
    df,
    on='row_index',
    how='left'
)

selected_df['True_Class'] = selected_df[
    'y_true'
].map(
    dict(enumerate(CLASS_NAMES))
)

selected_df['Predicted_Class'] = selected_df[
    'y_pred'
].map(
    dict(enumerate(CLASS_NAMES))
)

selected_df.to_csv(
    OUT
    / 'GradCAM_Selected_Examples.csv',
    index=False
)

print('Selected examples:')
display(
    selected_df[[
        'Case',
        'row_index',
        'True_Class',
        'Predicted_Class',
        'confidence',
        'true_probability',
        'filepath'
    ]]
)

assert len(selected_df) >= 4, (
    'STOP: Fewer than four Grad-CAM examples were selected.'
)

print('\n✅ CELL 5 COMPLETE')

In [ ]:
# ============================================================
# CELL 6 — GENERATE + SAVE INDIVIDUAL GRAD-CAM FIGURES
# ============================================================

result_rows = []

for _, row in selected_df.iterrows():

    case = row['Case']
    idx = int(row['row_index'])

    filepath = str(
        row['filepath']
    )

    true_id = int(
        y_true[idx]
    )

    pred_id = int(
        y_pred[idx]
    )

    confidence = float(
        probs[idx, pred_id]
    )

    # Grad-CAM is generated for the class actually predicted by the model.
    original, img_batch = (
        load_image_for_model(
            filepath
        )
    )

    heatmap = make_gradcam_heatmap(
        img_batch,
        pred_id
    )

    overlay = overlay_heatmap(
        original,
        heatmap,
        alpha=0.40
    )

    # Save raw numeric heatmap for reproducibility.
    np.save(
        OUT
        / f'{case}_heatmap.npy',
        heatmap
    )

    # Save standalone original.
    fig = plt.figure(
        figsize=(4.5, 4.5)
    )

    plt.imshow(original)
    plt.axis('off')
    plt.tight_layout()

    fig.savefig(
        OUT
        / f'{case}_original.png',
        dpi=300,
        bbox_inches='tight',
        pad_inches=0
    )

    plt.close(fig)

    # Save standalone overlay.
    fig = plt.figure(
        figsize=(4.5, 4.5)
    )

    plt.imshow(overlay)
    plt.axis('off')
    plt.tight_layout()

    fig.savefig(
        OUT
        / f'{case}_overlay.png',
        dpi=300,
        bbox_inches='tight',
        pad_inches=0
    )

    plt.close(fig)

    # Save publication-oriented three-panel figure.
    fig = plt.figure(
        figsize=(11, 3.7)
    )

    ax1 = fig.add_subplot(
        1, 3, 1
    )
    ax1.imshow(original)
    ax1.set_title('Original image')
    ax1.axis('off')

    ax2 = fig.add_subplot(
        1, 3, 2
    )
    ax2.imshow(
        heatmap,
        cmap='jet'
    )
    ax2.set_title(
        f'Grad-CAM\n{target_layer.name}'
    )
    ax2.axis('off')

    ax3 = fig.add_subplot(
        1, 3, 3
    )
    ax3.imshow(overlay)
    ax3.set_title(
        'Overlay\n'
        f'True: {CLASS_NAMES[true_id]} | '
        f'Pred: {CLASS_NAMES[pred_id]}\n'
        f'Predicted-class probability: {confidence:.4f}'
    )
    ax3.axis('off')

    fig.suptitle(
        case.replace('_', ' '),
        fontsize=12
    )

    fig.tight_layout()

    fig.savefig(
        OUT
        / f'{case}_GradCAM.pdf',
        bbox_inches='tight'
    )

    fig.savefig(
        OUT
        / f'{case}_GradCAM.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()
    plt.close(fig)

    result_rows.append({
        'Case': case,
        'row_index': idx,
        'filepath': filepath,
        'True_Class': CLASS_NAMES[
            true_id
        ],
        'Predicted_Class': CLASS_NAMES[
            pred_id
        ],
        'Predicted_Class_Probability':
            confidence,
        'GradCAM_Target_Layer':
            target_layer.name,
        'Heatmap_Min':
            float(
                np.min(heatmap)
            ),
        'Heatmap_Max':
            float(
                np.max(heatmap)
            ),
        'Heatmap_Mean':
            float(
                np.mean(heatmap)
            )
    })

gradcam_results = pd.DataFrame(
    result_rows
)

gradcam_results.to_csv(
    OUT
    / 'GradCAM_Final_Results.csv',
    index=False
)

print('\n✅ Individual Grad-CAM outputs saved')
print('✅ CELL 6 COMPLETE')

In [ ]:
# ============================================================
# CELL 7 — CREATE ONE 5-CASE OVERVIEW FIGURE
# ============================================================

n = len(selected_df)

fig = plt.figure(
    figsize=(11, 3.2 * n)
)

for r, (_, row) in enumerate(
    selected_df.iterrows()
):

    case = row['Case']
    idx = int(row['row_index'])

    filepath = str(
        row['filepath']
    )

    true_id = int(
        y_true[idx]
    )

    pred_id = int(
        y_pred[idx]
    )

    confidence = float(
        probs[idx, pred_id]
    )

    original, img_batch = (
        load_image_for_model(
            filepath
        )
    )

    heatmap = make_gradcam_heatmap(
        img_batch,
        pred_id
    )

    overlay = overlay_heatmap(
        original,
        heatmap,
        alpha=0.40
    )

    ax1 = fig.add_subplot(
        n, 3, 3*r + 1
    )
    ax1.imshow(original)
    ax1.axis('off')

    if r == 0:
        ax1.set_title(
            'Original'
        )

    ax2 = fig.add_subplot(
        n, 3, 3*r + 2
    )
    ax2.imshow(
        heatmap,
        cmap='jet'
    )
    ax2.axis('off')

    if r == 0:
        ax2.set_title(
            'Grad-CAM'
        )

    ax3 = fig.add_subplot(
        n, 3, 3*r + 3
    )
    ax3.imshow(overlay)
    ax3.axis('off')

    if r == 0:
        ax3.set_title(
            'Overlay'
        )

    ax1.set_ylabel(
        case.replace('_', ' '),
        fontsize=9
    )

    ax3.text(
        0.5,
        -0.06,
        (
            f'True: {CLASS_NAMES[true_id]} | '
            f'Pred: {CLASS_NAMES[pred_id]} | '
            f'p={confidence:.3f}'
        ),
        transform=ax3.transAxes,
        ha='center',
        va='top',
        fontsize=8
    )

fig.tight_layout()

fig.savefig(
    OUT
    / 'FINAL_MobileNetV2_GradCAM_Overview.pdf',
    bbox_inches='tight'
)

fig.savefig(
    OUT
    / 'FINAL_MobileNetV2_GradCAM_Overview.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

print('✅ Overview figure saved')
print('✅ CELL 7 COMPLETE')

In [ ]:
# ============================================================
# CELL 8 — SAVE METHODS / CAPTION DRAFT
# ============================================================

methods_text = (
    "Grad-CAM was applied to the final frozen-backbone "
    "MobileNetV2 baseline using the last spatial feature "
    f"representation ({target_layer.name}). For each selected "
    "test image, gradients of the predicted-class score with "
    "respect to the feature maps were globally averaged to "
    "obtain channel weights. A rectified weighted sum of the "
    "feature maps was normalized and resized to the input-image "
    "resolution before overlay. The visualisations were generated "
    "after model selection and were not used for training or "
    "hyperparameter selection."
)

caption_text = (
    "Grad-CAM visualisations for representative MobileNetV2 "
    "predictions on the leakage-controlled held-out test set. "
    "Each row shows the original image, Grad-CAM activation map, "
    "and overlay for the model-predicted class. Correct cataract "
    "and normal predictions are shown together with representative "
    "failure or difficult cases. Grad-CAM indicates regions that "
    "contributed to a prediction but should not be interpreted as "
    "proof of clinical causality."
)

(
    OUT
    / 'GradCAM_Methods_Draft.txt'
).write_text(
    methods_text
)

(
    OUT
    / 'GradCAM_Caption_Draft.txt'
).write_text(
    caption_text
)

print('METHODS DRAFT:')
print(methods_text)

print('\nCAPTION DRAFT:')
print(caption_text)

print('\n✅ CELL 8 COMPLETE')

In [ ]:
# ============================================================
# CELL 9 — FINAL COMPLETION CHECK
# ============================================================

required = [
    'GradCAM_Selected_Examples.csv',
    'GradCAM_Final_Results.csv',
    'FINAL_MobileNetV2_GradCAM_Overview.pdf',
    'FINAL_MobileNetV2_GradCAM_Overview.png',
    'GradCAM_Methods_Draft.txt',
    'GradCAM_Caption_Draft.txt'
]

for case in selected_df['Case']:

    required += [
        f'{case}_heatmap.npy',
        f'{case}_original.png',
        f'{case}_overlay.png',
        f'{case}_GradCAM.pdf',
        f'{case}_GradCAM.png'
    ]

missing = [
    fn
    for fn in required
    if not (OUT / fn).exists()
]

if missing:

    print('Missing:')
    for fn in missing:
        print('❌', fn)

    raise RuntimeError(
        'STOP: Grad-CAM outputs are incomplete.'
    )

(
    OUT
    / 'GRADCAM_DONE.txt'
).write_text(
    'Final MobileNetV2 Grad-CAM analysis completed.\n'
    f'Target layer: {target_layer.name}\n'
    f'Examples: {len(selected_df)}\n'
    'No training performed.\n'
)

print('========================================')
print('✅ FINAL MOBILENETV2 GRAD-CAM COMPLETE')
print('========================================')

print('\nSaved to:')
print(OUT)

print(
    '\nNEXT: Save/download this executed notebook '
    'and upload it to ChatGPT for visual interpretation.'
)

print(
    '\nDO NOT CHANGE THE MANUSCRIPT UNTIL '
    'THE HEATMAPS ARE VISUALLY CHECKED.'
)